# Phase 2 — YOLOv8s Fine-Tuning & ONNX Export

**Run on Google Colab with T4 GPU** (Runtime → Change runtime type → T4 GPU)

**Prerequisite:** `01_data_exploration.ipynb` must have been run — dataset saved to Drive at `TrafficVision/datasets/traffic_yolo/`

Steps:
1. Mount Drive & configure paths
2. Install dependencies
3. Verify GPU
4. Copy dataset to Colab SSD (fast I/O via parallel zip)
5. Fine-tune YOLOv8s (pretrained COCO → our 4-class traffic dataset)
6. Plot training curves
7. Evaluate — mAP, per-class metrics, confusion matrix, sample predictions
8. Export to ONNX
9. Verify ONNX inference speed
10. Save weights to Drive

### SSD copy strategy
- **First run ever**: Reads dataset from Drive using 32 parallel threads (much faster than sequential FUSE reads). Creates `traffic_yolo.zip` on Drive. Takes ~5-20 min once.
- **Every future session**: Copies the single zip file from Drive to SSD (~1-3 min) then extracts locally. No per-file FUSE latency.
- **Training**: `cache=False` — SSD reads are ~500 MB/s so no RAM cache needed.

## 1. Mount Google Drive

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_BACKEND_VERSION' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/TrafficVision'
else:
    _repo = os.path.abspath(os.path.join(os.getcwd(), '..'))
    DRIVE_BASE = os.path.join(_repo, 'data', 'TrafficVision')
    print('[LOCAL MODE] Using local data/ directory')

DATASET_YAML = os.path.join(DRIVE_BASE, 'datasets', 'traffic_yolo', 'dataset.yaml')
WEIGHTS_DIR  = os.path.join(DRIVE_BASE, 'weights')
RUNS_DIR     = '/content/runs' if IN_COLAB else os.path.join(_repo if not IN_COLAB else '/content', 'runs')

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

assert os.path.exists(DATASET_YAML), (
    f'dataset.yaml not found at {DATASET_YAML}\n'
    'Run 01_data_exploration.ipynb first to prepare the dataset.'
)

print(f'Drive base   : {DRIVE_BASE}')
print(f'Dataset yaml : {DATASET_YAML}')
print(f'Weights dir  : {WEIGHTS_DIR}')
print(f'Runs dir     : {RUNS_DIR}')

## 2. Install Dependencies

In [ ]:
!pip install -q ultralytics onnx onnxruntime-gpu

## 3. Verify GPU

In [ ]:
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU             : {props.name}')
    print(f'VRAM            : {props.total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU - enable GPU in Runtime > Change runtime type')
    print('Training will be extremely slow on CPU.')

## 4. Copy Dataset to Colab SSD

**Why not train directly from Drive?**  
Drive FUSE reads each small image file individually at ~0.2 MB/s with ~50ms per-file latency. 5800 images × 1.3 s/image = ~2 hours just for the RAM cache build every session.

**The fix**: Copy the whole dataset as a single zip file to Colab's NVMe SSD, then train directly from SSD at 500 MB/s with `cache=False`.

| Scenario | Time |
|----------|------|
| First run — parallel zip creation | ~5–20 min (once ever) |
| Future sessions — copy zip + extract | ~1–3 min |
| Training per epoch (SSD, cache=False) | ~3–5 min |

In [ ]:
import os, time, shutil, zipfile, threading, subprocess
import yaml as _yaml
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

LOCAL_EXTRACT  = '/content/traffic_yolo_local'  # extraction target
DRIVE_DATASETS = f'{DRIVE_BASE}/datasets'
DRIVE_ZIP      = f'{DRIVE_DATASETS}/traffic_yolo.zip'
LOCAL_ZIP      = '/content/traffic_yolo.zip'
SRC_DIR        = f'{DRIVE_DATASETS}/traffic_yolo'


def _find_dataset_root(base):
    """Find the directory under base that contains images/."""
    base = Path(base)
    if (base / "images").exists():
        return base
    for sub in sorted(base.iterdir()):
        if sub.is_dir() and (sub / "images").exists():
            return sub
    raise RuntimeError(
        f"Cannot find images/ under {base}.\n"
        f"Contents: {[p.name for p in base.iterdir()]}"
    )


if IN_COLAB:
    # Force Drive FUSE to populate directory listings before any exists() check.
    # Without this, os.path.exists() can return False on files that are there.
    subprocess.run(["ls", DRIVE_BASE],     capture_output=True)
    subprocess.run(["ls", DRIVE_DATASETS], capture_output=True)
    print(f"Drive datasets dir: {os.listdir(DRIVE_DATASETS) if os.path.isdir(DRIVE_DATASETS) else "NOT FOUND"}")

    # ── Step 1: get dataset onto local SSD ───────────────────────────
    if os.path.exists(LOCAL_EXTRACT):
        print('Dataset already on SSD — skipping copy.')

    elif os.path.exists(DRIVE_ZIP):
        # Validate local zip — delete if missing, empty, or corrupt
        _zip_ok = False
        if os.path.exists(LOCAL_ZIP) and os.path.getsize(LOCAL_ZIP) > 0:
            try:
                import zipfile as _zfmod
                with _zfmod.ZipFile(LOCAL_ZIP) as _ztest:
                    _zip_ok = len(_ztest.namelist()) > 0
            except Exception:
                _zip_ok = False
        if not _zip_ok:
            if os.path.exists(LOCAL_ZIP):
                print(f"Local zip invalid/corrupt — re-copying from Drive...")
                os.remove(LOCAL_ZIP)
            sz = os.path.getsize(DRIVE_ZIP) / 1e6
            print(f'Copying zip from Drive ({sz:.0f} MB) — ~1-3 min...')
            t0 = time.time()
            shutil.copy2(DRIVE_ZIP, LOCAL_ZIP)
            print(f'Copied in {time.time()-t0:.0f}s')
        else:
            print(f'Local zip valid — skipping Drive copy.')
        print('Extracting to SSD...')
        t0 = time.time()
        subprocess.run(['unzip', '-q', LOCAL_ZIP, '-d', LOCAL_EXTRACT], check=True)
        print(f'Extracted in {time.time()-t0:.0f}s')

    else:
        # One-time: parallel zip creation (32 threads overlap Drive FUSE latency)
        print('One-time setup: archiving dataset with 32 parallel threads...')
        print('(Happens ONCE — future sessions copy the zip in ~1-3 min)')
        src   = Path(SRC_DIR)
        files = sorted(f for f in src.rglob("*") if f.is_file())
        print(f'Found {len(files)} files to archive...')
        t0      = time.time()
        lock    = threading.Lock()
        counter = [0]

        def _archive_file(f):
            data    = f.read_bytes()
            arcname = str(f.relative_to(src))
            with lock:
                _zf.writestr(arcname, data)
                counter[0] += 1
                if counter[0] % 1000 == 0:
                    print(f'  {counter[0]}/{len(files)} files archived...')

        with zipfile.ZipFile(LOCAL_ZIP, "w", zipfile.ZIP_STORED) as _zf:
            with ThreadPoolExecutor(max_workers=32) as pool:
                futures = [pool.submit(_archive_file, f) for f in files]
                for fut in as_completed(futures):
                    fut.result()

        elapsed = time.time() - t0
        size_mb = os.path.getsize(LOCAL_ZIP) / 1e6
        print(f'Zip: {size_mb:.0f} MB in {elapsed:.0f}s ({elapsed/60:.1f} min)')
        print('Saving zip to Drive for future sessions...')
        shutil.copy2(LOCAL_ZIP, DRIVE_ZIP)
        print('Extracting to SSD...')
        subprocess.run(['unzip', '-q', LOCAL_ZIP, '-d', LOCAL_EXTRACT], check=True)
        print('Done.')

    # ── Step 2: find actual dataset root (handles zip subdirectory wrappers) ──
    _root = _find_dataset_root(LOCAL_EXTRACT)
    print(f'Dataset root: {_root}')

    # ── Step 3: write dataset.yaml pointing at the correct local root ──
    LOCAL_YAML = '/content/dataset.yaml'
    _yaml_src = _root / "dataset.yaml"
    if not _yaml_src.exists():
        _yaml_src = Path(SRC_DIR) / "dataset.yaml"
    with open(_yaml_src) as _f:
        _cfg = _yaml.safe_load(_f)

    _cfg["path"] = str(_root)  # absolute path to the correct root

    # Verify/fix split paths against what actually exists on disk
    _SPLIT_ALT = {
        'train': ['images/train', 'train'],
        'val':   ['images/val', 'images/valid', 'images/validation',
                  'val', 'valid', 'validation', 'images/test', 'test'],
        'test':  ['images/test', 'test', 'images/val', 'images/valid'],
    }
    for _split, _alts in _SPLIT_ALT.items():
        _declared = _cfg.get(_split)
        if not _declared:
            continue
        if (_root / _declared).exists():
            print(f"  {_split}: {_declared!r} OK")
        else:
            for _alt in _alts:
                if (_root / _alt).exists():
                    print(f"  {_split}: {_declared!r} -> {_alt!r}")
                    _cfg[_split] = _alt
                    break
            else:
                print(f"  WARNING: no directory found for {_split!r} under {_root}")

    # Final assertion — catch bad paths before training starts
    for _split in ("train", "val"):
        _p = _root / _cfg[_split]
        assert _p.exists(), (
            f"Split {_split!r} path does not exist: {_p}\n"
            f"Contents of {_root}: {[x.name for x in _root.iterdir()]}"
        )

    with open(LOCAL_YAML, 'w') as _f:
        _yaml.dump(_cfg, _f, default_flow_style=False)
    DATASET_YAML = LOCAL_YAML
    print(f'dataset.yaml OK — path={_root}, train={_cfg["train"]}, val={_cfg["val"]}')

else:
    print('[LOCAL] Skipping dataset copy')

## 5. Fine-tune YOLOv8s

Starting from `yolov8s.pt` (pretrained on COCO 80 classes) and fine-tuning on our 4-class traffic dataset. The pretrained backbone gives us strong feature extraction out of the box; fine-tuning adapts the head to our specific classes.

**`cache=False`**: Data is on local SSD (500 MB/s). No RAM cache needed — disk reads are ~instant compared to a GPU training step.

**Augmentation:** `degrees=0` (fixed cameras), `flipud=0` (no upside-down vehicles), `mosaic=1.0` (small/occluded vehicles), `scale=0.5` (distance variation)

**Checkpoints:** After every epoch, `last.pt` and `best.pt` are copied to Drive. If interrupted, the next run auto-resumes from Drive checkpoint.

In [ ]:
import os, shutil, subprocess
from ultralytics import YOLO

MODEL_NAME = 'yolov8s_traffic'
BEST_PT    = f'{RUNS_DIR}/{MODEL_NAME}/weights/best.pt'
LAST_PT    = f'{RUNS_DIR}/{MODEL_NAME}/weights/last.pt'
DRIVE_LAST = f'{WEIGHTS_DIR}/last.pt'
DRIVE_BEST = f'{WEIGHTS_DIR}/best.pt'

print(f'Training data : {DATASET_YAML}')
assert '/content/' in DATASET_YAML or not IN_COLAB, (
    'DATASET_YAML still points to Drive! Run cell-copy-dataset first.'
)

# Force Drive FUSE to sync before checking for checkpoints
if IN_COLAB:
    subprocess.run(["ls", WEIGHTS_DIR], capture_output=True)
    print(f'Weights dir contents: {os.listdir(WEIGHTS_DIR) if os.path.isdir(WEIGHTS_DIR) else "empty"}')

def save_to_drive(trainer):
    if not IN_COLAB:
        return
    if os.path.exists(LAST_PT):
        shutil.copy2(LAST_PT, DRIVE_LAST)
    if os.path.exists(BEST_PT):
        shutil.copy2(BEST_PT, DRIVE_BEST)
    if trainer.epoch % 5 == 0:
        print(f'[Drive] Checkpoints saved at epoch {trainer.epoch + 1}')

if os.path.exists(DRIVE_LAST):
    print(f'Resuming from {DRIVE_LAST}')
    model = YOLO(DRIVE_LAST)
    resume = True
else:
    print('Starting fresh from yolov8s.pt')
    model = YOLO('yolov8s.pt')
    resume = False

model.add_callback('on_train_epoch_end', save_to_drive)

results = model.train(
    data=DATASET_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    cache=False,     # data is on local SSD — reads are instant, no RAM cache needed
    device=0,
    workers=4,
    patience=15,
    save=True,
    save_period=10,
    project=RUNS_DIR,
    name=MODEL_NAME,
    exist_ok=True,
    resume=resume,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    plots=True,
    verbose=True,
)

## 6. Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

results_csv = os.path.join(RUNS_DIR, MODEL_NAME, 'results.csv')
assert os.path.exists(results_csv), f'results.csv not found at {results_csv}'

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

metrics = [
    ('train/box_loss',      'Train Box Loss',      '#e84c4c'),
    ('train/cls_loss',      'Train Class Loss',     '#e8a44c'),
    ('train/dfl_loss',      'Train DFL Loss',       '#8be84c'),
    ('metrics/mAP50(B)',    'Val mAP@0.5',          '#4c8be8'),
    ('metrics/mAP50-95(B)', 'Val mAP@0.5:0.95',    '#be84c8'),
    ('val/box_loss',        'Val Box Loss',          '#4ce8e8'),
]

for ax, (col, title, color) in zip(axes, metrics):
    if col in df.columns:
        ax.plot(df['epoch'], df[col], color=color, linewidth=2)
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)
        best_val = df[col].min() if 'loss' in col else df[col].max()
        best_ep  = df.loc[df[col].idxmax() if 'mAP' in col else df[col].idxmin(), 'epoch']
        ax.axvline(best_ep, color='red', linestyle='--', alpha=0.5,
                   label=f'Best: {best_val:.4f} @ ep{int(best_ep)}')
        ax.legend(fontsize=9)
    else:
        ax.set_title(f'{title} (not found)', fontsize=10)
        ax.axis('off')

plt.suptitle('Training Curves - YOLOv8s Traffic', fontsize=15, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(DRIVE_BASE, 'datasets', '05_training_curves.png')
plt.savefig(save_path, dpi=120, bbox_inches='tight')
plt.show()
print('Saved:', save_path)

## 7. Evaluate on Validation Set

In [ ]:
import os
from ultralytics import YOLO

assert os.path.exists(BEST_PT), f'Best weights not found: {BEST_PT}'

model   = YOLO(BEST_PT)
metrics = model.val(
    data=DATASET_YAML,
    imgsz=640,
    device=0,
    plots=True,
    save_json=False,
)

CLASS_NAMES = ['car', 'bus', 'motorcycle', 'truck']

print()
print('=' * 45)
print(f'  Overall mAP@0.5     : {metrics.box.map50:.4f}')
print(f'  Overall mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'  Precision           : {metrics.box.mp:.4f}')
print(f'  Recall              : {metrics.box.mr:.4f}')
print('=' * 45)
print('  Per-class AP@0.5:')
for i, (name, ap) in enumerate(zip(CLASS_NAMES, metrics.box.ap50)):
    print(f'    {name:<12}: {ap:.4f}')
print('=' * 45)

### Per-class AP

In [ ]:
import matplotlib.pyplot as plt

ALL_CLASS_NAMES  = ['car', 'bus', 'motorcycle', 'truck']
ALL_CLASS_COLORS = ['#4c8be8', '#e8a44c', '#e84c4c', '#8be84c']
ap50_vals = list(metrics.box.ap50)

# Use actual class names from the model if available; fall back to slicing
# the hard-coded list.  This guards against a class having no val instances
# (metrics.box.ap50 will then have fewer entries than CLASS_NAMES).
if hasattr(metrics, 'names') and metrics.names:
    CLASS_NAMES  = [metrics.names[i] for i in range(len(ap50_vals))]
    CLASS_COLORS = ALL_CLASS_COLORS[:len(ap50_vals)]
else:
    CLASS_NAMES  = ALL_CLASS_NAMES[:len(ap50_vals)]
    CLASS_COLORS = ALL_CLASS_COLORS[:len(ap50_vals)]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(CLASS_NAMES, ap50_vals, color=CLASS_COLORS, edgecolor='none', width=0.5)
ax.set_ylim(0, 1.05)
ax.set_ylabel('AP @ IoU=0.50', fontsize=12)
ax.set_title(f'Per-Class AP@0.5  |  mAP@0.5 = {metrics.box.map50:.4f}', fontsize=13)
ax.axhline(metrics.box.map50, color='red', linestyle='--', linewidth=1.5,
           label=f'mAP@0.5 = {metrics.box.map50:.4f}')
ax.legend()
for bar, val in zip(bars, ap50_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(DRIVE_BASE, 'datasets', '06_per_class_ap.png')
plt.savefig(save_path, dpi=120, bbox_inches='tight')
plt.show()
print('Saved:', save_path)

### Sample Predictions

In [ ]:
import cv2, random, yaml
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

with open(DATASET_YAML) as f:
    ds_cfg = yaml.safe_load(f)

val_img_dir = Path(ds_cfg['path']) / ds_cfg['val']
val_imgs    = sorted(val_img_dir.glob('*.jpg'))
sample_imgs = random.sample(val_imgs, min(6, len(val_imgs)))

CLASS_NAMES = ['car', 'bus', 'motorcycle', 'truck']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_imgs):
    result  = model(img_path, verbose=False, conf=0.3)[0]
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    for box in result.boxes:
        cls_id = int(box.cls)
        conf   = float(box.conf)
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        color = ['#6464ff', '#ffb464', '#64ff64', '#ffc864'][cls_id % 4]
        rect  = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                   linewidth=1.5, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 3, f'{CLASS_NAMES[cls_id]} {conf:.2f}',
                color=color, fontsize=7, fontweight='bold')
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

plt.suptitle('Sample Predictions on Validation Set (conf > 0.3)', fontsize=14, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(DRIVE_BASE, 'datasets', '07_sample_predictions.png')
plt.savefig(save_path, dpi=120, bbox_inches='tight')
plt.show()
print('Saved:', save_path)

## 8. Export to ONNX

- `opset=17` — compatible with ONNX Runtime 1.16+
- `simplify=True` — removes redundant ops
- `dynamic=False` — fixed batch=1, better GPU optimization
- `half=False` — FP32 for maximum compatibility

In [ ]:
from ultralytics import YOLO
import os

model = YOLO(BEST_PT)

export_path = model.export(
    format='onnx',
    imgsz=640,
    opset=17,
    simplify=True,
    dynamic=False,
    half=False,
)

ONNX_PATH = str(export_path)
print(f'ONNX exported to: {ONNX_PATH}')

size_mb = os.path.getsize(ONNX_PATH) / 1e6
print(f'File size: {size_mb:.1f} MB')
assert size_mb > 5, 'ONNX file suspiciously small - export may have failed'

## 9. Verify ONNX Inference

In [ ]:
import onnxruntime as ort
import numpy as np
import time

providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
sess      = ort.InferenceSession(ONNX_PATH, providers=providers)

print(f'Providers active : {sess.get_providers()}')
print(f'Input  name      : {sess.get_inputs()[0].name}')
print(f'Input  shape     : {sess.get_inputs()[0].shape}')
print(f'Output shape     : {sess.get_outputs()[0].shape}')

dummy      = np.random.rand(1, 3, 640, 640).astype(np.float32)
input_name = sess.get_inputs()[0].name

for _ in range(3):
    sess.run(None, {input_name: dummy})

N = 100
t0 = time.perf_counter()
for _ in range(N):
    sess.run(None, {input_name: dummy})
elapsed = time.perf_counter() - t0

print(f'\nBenchmark ({N} frames):')
print(f'  Latency : {elapsed / N * 1000:.1f} ms/frame')
print(f'  FPS     : {N / elapsed:.1f}')
print(f'\nONNX inference verified.')

## 10. Save Weights to Google Drive

In [ ]:
import shutil
from pathlib import Path

src_pt   = Path(BEST_PT)
src_onnx = Path(ONNX_PATH)

dst_pt   = Path(WEIGHTS_DIR) / 'yolov8s_traffic_best.pt'
dst_onnx = Path(WEIGHTS_DIR) / 'yolov8s_traffic.onnx'

shutil.copy2(src_pt,   dst_pt)
shutil.copy2(src_onnx, dst_onnx)

print('Saved to Google Drive:')
print(f'  PT   : {dst_pt}   ({dst_pt.stat().st_size / 1e6:.1f} MB)')
print(f'  ONNX : {dst_onnx} ({dst_onnx.stat().st_size / 1e6:.1f} MB)')

val_plots_dir = Path(RUNS_DIR) / MODEL_NAME
for png in val_plots_dir.glob('*.png'):
    dst = Path(DRIVE_BASE) / 'datasets' / png.name
    shutil.copy2(png, dst)
    print(f'  Plot : {dst.name}')

print('\nAll weights and plots saved to Drive.')

## Done — Phase 2 Complete

### What was produced

| File | Location on Drive |
|------|-------------------|
| Best PyTorch weights | `TrafficVision/weights/yolov8s_traffic_best.pt` |
| ONNX model | `TrafficVision/weights/yolov8s_traffic.onnx` |
| Training curves | `TrafficVision/datasets/05_training_curves.png` |
| Per-class AP | `TrafficVision/datasets/06_per_class_ap.png` |
| Sample predictions | `TrafficVision/datasets/07_sample_predictions.png` |
| Confusion matrix | `TrafficVision/datasets/confusion_matrix.png` |

### Next steps

1. **Download the ONNX model** from Google Drive to your local machine:
   ```
   models/weights/yolov8s_traffic.onnx
   ```
2. **Start Phase 3** — build the core inference engine:
   - `core/detector.py` — ONNX Runtime wrapper
   - `core/tracker.py` — ByteTrack via supervision
   - `core/analytics.py` — counting, speed, anomalies
   - `core/pipeline.py` — orchestrator